# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHIT-25607/FLYRANK-INTERN/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Lane: **Refresh / Content Opportunity Scoring** (from Week 1). This notebook writes the data
contract for that lane against the full warehouse release: what a row means, which tables and
windows I use, what I predict, what I exclude — then proves three facts with real queries on a
mid-panel month, builds five features, and springs the leakage trap on purpose so I can see it
and delete it.

Careful words only: observed, measured, directional, decision-support.

## 0. Access + single scan (the skill's iteration rule)

The warehouse is a gated Hugging Face dataset. The token is read from **environment variable
`HF_TOKEN` -> Colab Secret -> prompt**, never hard-coded (the repo is public). Then I read the
**two month partitions I need once each** — decision month `month=2026-03` and label month
`month=2026-04` — into in-memory DuckDB tables. Every query below runs on that cached slice, so
the heavy network scan happens exactly once, per the iteration rule in the data skill.

In [1]:
%pip -q install duckdb huggingface_hub
print("deps ok")

Note: you may need to restart the kernel to use updated packages.
deps ok



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, getpass
import duckdb, pandas as pd
import numpy as np

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 140)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET http_timeout = 900")

REL = "hf://datasets/FlyRank/internship-warehouse"
SRC_MAR = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
SRC_APR = f"{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet"
SRC = lambda t: f"read_parquet('{REL}/{t}.parquet')"
print("connected to", REL)

connected to hf://datasets/FlyRank/internship-warehouse


In [3]:
# One scan per month. Only the columns this lane touches.
MAR_COLS = ("report_date, client_hash_id, content_hash_id, gsc_data_available, "
            "ga4_data_available, gsc_impressions, gsc_clicks, gsc_avg_position, month")
con.execute(f"CREATE OR REPLACE TABLE fact_march AS SELECT {MAR_COLS} FROM read_parquet('{SRC_MAR}')")
con.execute(f"CREATE OR REPLACE TABLE fact_april AS "
            f"SELECT report_date, client_hash_id, content_hash_id, gsc_data_available, gsc_impressions "
            f"FROM read_parquet('{SRC_APR}')")
print("fact_march rows:", f"{con.sql('SELECT COUNT(*) FROM fact_march').fetchone()[0]:,}")
print("fact_april rows:", f"{con.sql('SELECT COUNT(*) FROM fact_april').fetchone()[0]:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_march rows: 9,841,378
fact_april rows: 10,424,730


## 1. Unit of analysis + time window

**The contract, in plain words (the 5 answers):**

1. **One row means** — in `fact_content_daily_performance` one row is one **content item (page)
   × one pseudonymized client × one report_date**, i.e. a **page-day** of search data. My lane's
   unit of analysis is the **page**: I fold a page's March page-days into one feature row (that is
   the row an editor's queue is built from).
2. **Which tables** — `fact_content_daily_performance` is the workhorse: the `month=2026-03`
   partition for features, the `month=2026-04` partition for the observed label. `dim_clients`
   and `dim_content` are used as context only (coverage, joins, filters).
3. **Time window** — **features: 2026-03-01 → 03-31** (the whole decision month, fully closed when
   the queue is built); **label: the next 30 days, 2026-04-01 → 04-30** (observed later). Feature
   and label windows are adjacent and do not overlap.
4. **What I'd predict / rank** — an **observed** binary label `declined_next_30d = 1` when a
   page's April impressions are < 80% of its March impressions. I rank pages by predicted
   probability of that label and surface the top K as the editor's review queue. On this slice the
   label is a genuine prior→future outcome, unlike the starter's same-window
   `is_declining_label` proxy.
5. **One thing I deliberately exclude** — GA4 engagement columns (`ga4_*`) from features: too
   sparse (~4.2% of March rows are flagged available) and zero-filled where `ga4_data_available
   IS NOT TRUE`, so they are currently un-trustworthy as signals (evidence below).

The dataframe below is the unit, raw: one row per page-day.

In [4]:
con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position
    FROM fact_march
    WHERE gsc_data_available IS TRUE
    LIMIT 6
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280


## 2. Fields: feature / label / context / excluded

Every field I touch goes in exactly one bucket:

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (aggregated per page into `imp_march`, `clk_march`, `pos_march`, `days_active_march`, `momentum_last7`) | knowable before the decision moment (end of March), measured, not future |
| **Label / proxy** | `declined_next_30d` (built from April `gsc_impressions`) | the thing I predict — observed in a **later** window; never a feature |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `client_has_gsc/_ga4`, `gsc_data_available` (filter), `month`, `dim_*` tables | grouping / joining / splitting / filtering only; IDs are never learned from |
| **Excluded** | `ga4_*` engagement columns; `gsc_sum_position`; `fact_content_query_90d`; provider/model metadata | see reasons below |

Exclusion reasons: **GA4** is sparse and zero-filled where unavailable (missingness follows an
availability flag, not randomness); **`gsc_sum_position`** restates `gsc_avg_position`;
**`fact_content_query_90d`** has a fixed 90-day window that overlaps the label months — I would be
labelling inside my own feature window until I align it, so it stays out this week; provider/model
metadata describe how a page was written, not how it performs.

In [5]:
# Missingness follows the flags. GA4 is nearly empty; gsc_data_available here is TRUE/FALSE,
# while ga4_data_available is three-valued (TRUE / FALSE / <NA>) -> must filter with IS TRUE.
gsc_pos_zero = con.sql(
    "SELECT COUNT(*) FROM fact_march WHERE gsc_data_available IS TRUE AND gsc_avg_position = 0"
).fetchone()[0]
print(f"gsc_avg_position = 0 means 'no position data', not rank 0: {gsc_pos_zero:,} valid-GSC rows in March")

flag_counts = con.sql("""
    SELECT gsc_data_available, ga4_data_available, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM fact_march), 2) AS pct
    FROM fact_march
    GROUP BY 1, 2 ORDER BY 3 DESC
""").df()
flag_counts

gsc_avg_position = 0 means 'no position data', not rank 0: 163,189 valid-GSC rows in March


,gsc_data_available,ga4_data_available,n,pct
0,False,False,4690323,47.66
1,True,False,1718348,17.46
2,True,<NA>,1528366,15.53
3,False,<NA>,1490375,15.14
4,True,True,364347,3.70
5,False,True,49619,0.50


## 3. Verify it with queries (three facts on `month=2026-03`)

Every contract claim gets a query. Three facts, exactly:

**Fact 1 — the grain holds.** *A contract line without a query next to it is a guess.*

In [6]:
# FACT 1: grain — one row really is one page-day. Zero grouped rows with COUNT > 1 proves it.
grain = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM fact_march
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("rows violating the page-day grain:", len(grain))
grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows violating the page-day grain: 0


,report_date,client_hash_id,content_hash_id,n


**Fact 2 — my slice's row count and date span.**

In [7]:
# FACT 2: count + date span for month=2026-03
summary = con.sql("""
    SELECT COUNT(*)                        AS page_day_rows,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS pages
    FROM fact_march
""").df()
summary

,page_day_rows,first_day,last_day,clients,pages
0,9841378,2026-03-01,2026-03-31,55,331437


**Fact 3 — availability, filtered with `IS TRUE`.** How many rows survive each flag. GSC is my
feature source and it survives ~37%; GA4 survives ~4% — which is exactly why it is excluded.

In [8]:
# FACT 3: availability — filter with IS TRUE, show survival and share.
avail = con.sql("""
    SELECT COUNT(*)                                              AS base_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)    AS gsc_IS_TRUE,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)    AS ga4_IS_TRUE,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                       AND  ga4_data_available IS TRUE)          AS both_IS_TRUE
    FROM fact_march
""").df()
av = avail.iloc[0]
print(f"survival: {100*av['gsc_IS_TRUE']/av['base_rows']:.1f}% gsc  |  "
      f"{100*av['ga4_IS_TRUE']/av['base_rows']:.1f}% ga4  |  "
      f"{100*av['both_IS_TRUE']/av['base_rows']:.1f}% both")
avail

survival: 36.7% gsc  |  4.2% ga4  |  3.7% both


,base_rows,gsc_IS_TRUE,ga4_IS_TRUE,both_IS_TRUE
0,9841378,3611061,413966,364347


### 3b. Five features (max) — and when each is knowable

Feature frame: one row per page, aggregated from **March only** (`gsc_data_available IS TRUE`),
for pages with measurable demand. Each feature's *available-when* line:

| # | Feature | Meaning | Knowable at the decision moment because… |
|---|---|---|---|
| 1 | `imp_march` | total GSC impressions in March | the month is fully closed when the queue is built on 03-31; the daily sync has already run its last March day |
| 2 | `clk_march` | total GSC clicks in March | same closed-window argument as above |
| 3 | `pos_march` | mean `gsc_avg_position` on valid-position days | position is measured per synced day, all ≤ 03-31; rows with the `=0` no-data sentinel are skipped |
| 4 | `days_active_march` | distinct days the page drew ≥ 1 impression | counts the page's already-synced March rows — fully known on 03-31 |
| 5 | `momentum_last7` | share of March clicks landing in 03-25→03-31 | the last 7 days are the most recent fully completed days; still strictly before the April label window |

None of these reads a column from April — the label window stays untouched.

In [9]:
feature_frame = con.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions)                                                  AS imp_march,
           SUM(gsc_clicks)                                                       AS clk_march,
           AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0)             AS pos_march,
           COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)        AS days_active_march,
           SUM(gsc_clicks) FILTER (WHERE report_date >= DATE '2026-03-25')
               * 1.0 / NULLIF(SUM(gsc_clicks), 0)                                AS momentum_last7
    FROM fact_march
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
""").df()

print("feature frame:", f"{len(feature_frame):,}", "pages x", len(feature_frame.columns),
      "columns (one page per row)")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feature frame: 176,738 pages x 7 columns (one page per row)


,client_hash_id,content_hash_id,imp_march,clk_march,pos_march,days_active_march,momentum_last7
0,client_73cda7b4e4f265ea,content_f8df6b20d18c4374,798.0,1.0,6.638543,31,0.000000
1,client_73cda7b4e4f265ea,content_5e94eb562b984cb5,2650.0,12.0,2.098693,31,0.083333
2,client_73cda7b4e4f265ea,content_cb0d77f160e8b890,1698.0,0.0,7.043618,31,NaN
3,client_73cda7b4e4f265ea,content_912d9a26c7e1375e,426.0,1.0,15.920932,31,0.000000
4,client_73cda7b4e4f265ea,content_c4524733c28b33ed,640.0,0.0,14.796462,31,NaN


### 3c. The trap — a label-derived column, added on purpose

Now the label: `declined_next_30d` observed from **April**. Then I deliberately add **one
label-derived column** — `apr_imp`, the raw April impressions the label is built from — and watch
the score jump toward perfect. That jump is *why* the honest windows matter: a model that 'sees'
the label's own inputs looks flawless even though it would predict nothing useful at the real
decision moment. Then I delete the column and keep the honest numbers.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

label = con.sql("""
    SELECT content_hash_id, SUM(gsc_impressions) AS apr_imp
    FROM fact_april
    WHERE gsc_data_available IS TRUE
    GROUP BY 1
""").df()

lane = feature_frame.merge(label, on="content_hash_id", how="inner")
lane = lane[lane["imp_march"] >= 100].reset_index(drop=True)   # measurable demand, mirror of starter pool
lane["declined_next_30d"] = (lane["apr_imp"] < 0.8 * lane["imp_march"]).astype(int)
print("lane pool:", f"{len(lane):,}", "pages | decline base rate:", round(lane["declined_next_30d"].mean(), 3))

honest_cols = ["imp_march", "clk_march", "pos_march", "days_active_march", "momentum_last7"]
md = lane.dropna(subset=honest_cols).reset_index(drop=True)
y = md["declined_next_30d"]

def score_and_rank(X, y, seed=42):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
    m = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1).fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, p)
    top = p.argsort()[::-1][:50]
    return round(auc, 3), round(yte.to_numpy()[top].mean(), 3), yte.mean()

auc_h, p50_h, base = score_and_rank(md[honest_cols], y)            # honest: 5 features only
auc_l, p50_l, _    = score_and_rank(md[honest_cols + ["apr_imp"]], y)   # trap: + 1 label-derived column
print(f"trap score jump: ROC-AUC {auc_h} -> {auc_l} | precision@50 {p50_h} -> {p50_l}")
print("-> a column derived from the label window makes the queue look near-perfect. Deleting it now.")

pd.DataFrame({
    "run":                    ["honest (5 features)", "trap (5 + apr_imp)", "random / base rate"],
    "ROC-AUC":                [auc_h, auc_l, 0.5],
    "precision@50 (top queue)": [p50_h, p50_l, round(base, 3)],
})

lane pool: 100,893 pages | decline base rate: 0.515


trap score jump: ROC-AUC 0.658 -> 0.999 | precision@50 0.56 -> 1.0
-> a column derived from the label window makes the queue look near-perfect. Deleting it now.


,run,ROC-AUC,precision@50 (top queue)
0,honest (5 features),0.658,0.560
1,trap (5 + apr_imp),0.999,1.000
2,random / base rate,0.500,0.473


In [11]:
# Delete the leak column, keep the honest number.
honest = lane.drop(columns=["apr_imp"])
print("leak column present in the final frame?", "apr_imp" in honest.columns)
print("final feature columns:", [c for c in honest.columns
      if c not in ("client_hash_id", "content_hash_id", "declined_next_30d")])

md_h = honest.dropna(subset=honest_cols).reset_index(drop=True)
auc_f, p50_f, base_f = score_and_rank(md_h[honest_cols], md_h["declined_next_30d"])
print(f"honest after deleting the leak -> ROC-AUC {auc_f} | precision@50 {p50_f} | base rate {round(base_f, 3)}")
honest.head()

leak column present in the final frame? False
final feature columns: ['imp_march', 'clk_march', 'pos_march', 'days_active_march', 'momentum_last7']


honest after deleting the leak -> ROC-AUC 0.658 | precision@50 0.56 | base rate 0.473


,client_hash_id,content_hash_id,imp_march,clk_march,pos_march,days_active_march,momentum_last7,declined_next_30d
0,client_73cda7b4e4f265ea,content_f8df6b20d18c4374,798.0,1.0,6.638543,31,0.000000,1
1,client_73cda7b4e4f265ea,content_5e94eb562b984cb5,2650.0,12.0,2.098693,31,0.083333,0
2,client_73cda7b4e4f265ea,content_cb0d77f160e8b890,1698.0,0.0,7.043618,31,NaN,0
3,client_73cda7b4e4f265ea,content_912d9a26c7e1375e,426.0,1.0,15.920932,31,0.000000,0
4,client_73cda7b4e4f265ea,content_c4524733c28b33ed,640.0,0.0,14.796462,31,NaN,1


## 4. Data limits

**One named limitation of my slice:** only a minority of clients have usable search data in the
decision month — **47 of 104 clients** have `gsc_data_available IS TRUE` rows in March, and even
among promised-history clients (52 with `gsc_data_start <= 2026-03-01`) 47 show up. The slice is
silently client-pruned, so findings describe a well-instrumented subset, not all clients.

Three more, one line each:
- **History depth is unbalanced** — per-client GSC start dates differ by over a year; any global
  calendar window is unequal across clients.
- **The label treats absence as missing, not outcome** — a page absent from April tracking is
  excluded (na), not counted as declined or stable, because the fact table only accrues tracked
  rows.
- **Direction, not cause** — matching March features to an April impressions drop shows
  correlation, not that refreshing any flagged page *causes* a recovery; that needs a controlled
  experiment this data cannot provide.

The code below backs the headline limitation.

In [12]:
dcl = con.sql(f"""SELECT client_hash_id, gsc_data_start FROM {SRC('dim_clients')}""").df()
promised = int((dcl["gsc_data_start"] <= "2026-03-01").sum())
present  = con.sql("SELECT COUNT(DISTINCT client_hash_id) FROM fact_march WHERE gsc_data_available IS TRUE").fetchone()[0]
print(f"clients with usable GSC rows in March: {present} of {promised} promised (of {len(dcl)} total)")

print("\nhistory depth spread (gsc_data_start):")
print(dcl["gsc_data_start"].describe().to_string())

clients with usable GSC rows in March: 47 of 52 promised (of 104 total)

history depth spread (gsc_data_start):
count                            67
mean     2025-11-17 00:42:59.104477
min             2025-01-27 00:00:00
25%             2025-09-24 00:00:00
50%             2025-11-05 00:00:00
75%             2026-02-19 00:00:00
max             2026-06-02 00:00:00


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (IDs are pseudonyms; the token lives in an env/secret, never a cell)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card. Done.